# Exercice 3 : l'API, recevoir les alertes

**Mission :** brancher le mini-SOAR sur les outils de détection. Les alertes arrivent par **HTTP**, dans des formats différents, parfois en double ; le SOAR les vérifie, les traduit, les trie et lance le bon playbook. Puis on rejoue **toute la matinée** et on obtient le premier score.

```
 SIEM  ─┐
        ├─► API (webhook) ──► moteur ──► actions
 Wazuh ─┘    ex. 3 ◄── vous êtes ici
```

**Ouvrez `minisoar/api.py`.** C'est une application **FastAPI**, comme l'API de Tracecat.

⚠️ Le rejeu utilise votre `playbooks/force-brute.yaml` de l'exercice 2. S'il n'est pas terminé, décommentez la ligne `shutil.copy` ci-dessous.

In [1]:
import shutil
# shutil.copy("solutions/playbooks/force-brute.yaml", "playbooks/force-brute.yaml")
import os
from fastapi.testclient import TestClient
from banc import scenario, environnement as env, rejeu
from minisoar import api

alertes = scenario.construire()
os.environ["SOAR_SECRET_WEBHOOK"] = "cle-du-siem-8inf857"     # la clé que les outils envoient

## 1. Un format commun, validé

Chaque outil envoie ses alertes à sa façon. Le SOAR les traduit dans **un seul** format, le modèle `Alerte`, et refuse ce qui n'y entre pas. Une alerte est une donnée **venue de l'extérieur** : on la valide comme une entrée de formulaire web.

In [2]:
print(api.normaliser("siem", alertes[5]))

for faux in [dict(alertes[5], compte="webadmin; rm -rf /"),
             dict(alertes[5], niveau=99),
             dict(alertes[5], ip="203.0.113.777")]:
    try:
        api.normaliser("siem", faux)
    except Exception as e:
        print("Refusée :", str(e).splitlines()[1], "→", str(e).splitlines()[2].strip()[:70])

id='A06' horodatage=datetime.datetime(2016, 12, 10, 8, 41, 12) regle='R2-succes-apres-echecs' niveau=7 hote='LabSZ' ip=IPv4Address('203.0.113.77') compte='webadmin' description='Connexion SSH réussie pour webadmin après 9 échecs depuis 203.0.113.77'
Refusée : compte → String should match pattern '^[a-z_][a-z0-9_-]{0,31}$' [type=string_pa
Refusée : niveau → Input should be less than or equal to 15 [type=less_than_equal, input_
Refusée : ip → value is not a valid IPv4 or IPv6 address [type=ip_any_address, input_


## 2. Normaliser : deux outils, deux formats

Les alertes « serveur » (sortant suspect, CPU) viennent de **Wazuh**, avec le format de Wazuh. Comparez :

In [3]:
source, brut = scenario.a_envoyer(alertes[10])
print(source, "envoie :", brut)
print("\nAprès normalisation :", api.normaliser(source, brut))

wazuh envoie : {'id': 'A11', 'timestamp': '2016-12-10T09:20:00.000+0000', 'rule': {'id': '100310', 'level': 6, 'description': 'CPU > 95 % depuis plus de 30 min sur LabSZ (processus « kworkerds », utilisateur « webadmin »)'}, 'agent': {'id': '007', 'name': 'LabSZ'}, 'data': {'dstuser': 'webadmin'}}

Après normalisation : id='A11' horodatage=datetime.datetime(2016, 12, 10, 9, 20) regle='R5-cpu' niveau=6 hote='LabSZ' ip=None compte='webadmin' description='CPU > 95 % depuis plus de 30 min sur LabSZ (processus « kworkerds », utilisateur « webadmin »)'


❓ **Question pour la classe :** pourquoi ne pas écrire les playbooks directement avec les champs de Wazuh (`data.srcip`, `rule.level`) ?

*Réponse : parce qu'on aurait un playbook par outil, et qu'il faudrait tous les réécrire le jour où on change de SIEM. On traduit **une fois**, à l'entrée ; tout le reste du SOAR ne connaît qu'un format. C'est l'idée des schémas **ECS** (Elastic) et **OCSF** (AWS, Splunk…).*

## 3. L'API, appelée comme le ferait le SIEM

**Ce qu'on fait :** `TestClient` appelle l'API **dans ce notebook**, sans lancer de serveur ; c'est exactement la même requête HTTP qu'enverrait le SIEM.

In [4]:
siem = TestClient(api.app)

r = siem.post("/webhooks/siem", json=alertes[0])
print(r.status_code, r.json())

r = siem.post("/webhooks/siem", json=alertes[0], headers={"x-soar-cle": "cle-du-siem-8inf857"})
print(r.status_code, r.json())

401 {'detail': "clé d'API absente ou invalide"}
200 {'alerte': 'A01', 'playbook': 'force-brute', 'execution': '2ef6e4a6', 'statut': 'terminée', 'sortie': {'verdict': 'attaque sans impact', 'rapport': '112.95.230.3 : 5 échecs de Dec 10 07:27:50 à Dec 10 07:28:03, 0 connexion(s) réussie(s). Blocage : réussie.'}}


👀 **Ce qu'on doit voir :** **401** sans la clé, puis **200** avec : l'alerte a été routée vers `force-brute`, le playbook a tourné, l'IP du robot est bloquée.

**Ce qu'on fait :** le SIEM renvoie la même alerte 30 secondes plus tard (c'est fréquent).

In [5]:
copie = dict(alertes[0], id="A01-bis", horodatage="2016-12-10T07:28:33")
print(siem.post("/webhooks/siem", json=copie, headers={"x-soar-cle": "cle-du-siem-8inf857"}).json())

{'alerte': 'A01-bis', 'statut': 'doublon ignoré', 'empreinte': 'ccdc9e9263881568'}


❓ **Question pour la classe :** la fenêtre de déduplication est de 10 minutes. Un attaquant qui le sait peut-il en profiter ?

*Réponse : oui. Une première alerte « bénigne » ouvre la fenêtre ; pendant 10 minutes, les alertes de même empreinte (même règle, même IP, même compte) sont ignorées. C'est pour ça que l'empreinte doit être assez **précise**, et qu'un doublon est ignoré mais **compté et journalisé**, jamais effacé.*

**Ce qu'on fait :** relire une exécution par l'API, comme dans l'onglet « Runs » de Tracecat.

In [6]:
id_ = r.json()["execution"]
siem.get(f"/executions/{id_}").json()["etapes"]

[{'ref': 'bilan',
  'action': 'journaux.connexions',
  'heure': '07:28:03',
  'entrees': {'ip': '112.95.230.3'},
  'result': {'echecs': 5,
   'succes': 0,
   'comptes_essayes': ['root'],
   'comptes_entres': [],
   'premier': 'Dec 10 07:27:50',
   'dernier': 'Dec 10 07:28:03'},
  'error': None,
  'essais': 1,
  'duree': 0.05,
  'statut': 'réussie'},
 {'ref': 'bloquer',
  'action': 'pare_feu.bloquer_ip',
  'heure': '07:28:03',
  'entrees': {'ip': '112.95.230.3'},
  'result': {'ip': '112.95.230.3', 'statut': 'bloquée'},
  'error': None,
  'essais': 1,
  'duree': 0.0,
  'statut': 'réussie'}]

## 4. Rejouer toute la matinée par HTTP

**Ce qu'on fait :** le banc d'essai envoie les 11 alertes dans l'ordre, chacune par son webhook et dans son format, puis **note** le SOAR : verdicts, faits trouvés, actions vérifiées dans l'environnement.

In [7]:
api.VUES.clear()

def envoyer(alerte):
    source, brut = scenario.a_envoyer(alerte)
    r = siem.post(f"/webhooks/{source}", json=brut, headers={"x-soar-cle": "cle-du-siem-8inf857"}).json()
    sortie = r.get("sortie") or {}
    return f"{sortie.get('rapport', r)}\nVERDICT : {sortie.get('verdict', '?')}"

score = rejeu.rejouer(envoyer, "3 - API et playbooks")

07:28:03 A01 R1-force-brute           → attaque sans impact  ✅  (0 s)
07:34:10 A02 R1-force-brute           → attaque sans impact  ✅  (0 s)
08:02:41 A03 R2-succes-apres-echecs   → faux positif         ✅  (0 s)


08:21:07 A04 R3-compte-leurre         → incident             ✅  (0 s)
08:25:15 A05 R1-force-brute           → attaque sans impact  ✅  (0 s)


08:41:12 A06 R2-succes-apres-echecs   → incident             ✅  (0 s)
09:05:00 A07 R4-sortant-suspect       → incident             ✅  (0 s)
09:09:42 A08 R1-force-brute           → attaque sans impact  ✅  (0 s)


09:11:34 A09 R1-force-brute           → attaque sans impact  ✅  (0 s)
09:13:10 A10 R1-force-brute           → attaque sans impact  ✅  (0 s)
09:20:00 A11 R5-cpu                   → faux positif         ❌  (0 s)

Score : 80/100   (verdicts 32/35 · faits 30/35 · réponse 18/30 · pénalités −0)
Faits trouvés : ✅ début de l'attaque à 07:05 · ✅ script x.sh téléchargé · ✅ clé SSH de l'attaquant ajoutée · ✅ crontab de l'attaquant ajoutée · ✅ binaire /tmp/.x/kworkerds · ✅ reconnaissance (uname, whoami, /etc/passwd) · ❌ source de l'injection (198.51.100.23) signalée
Actions : ✅ IP de l'attaquant bloquée · ✅ compte webadmin désactivé · ❌ clé SSH de l'attaquant retirée · ❌ crontab de l'attaquant retirée · ✅ mineur arrêté
Rapport complet : resultats/rapport - 3 - API et playbooks.md   (0.0 min)


👀 **Ce qu'on doit voir :** 10 verdicts sur 11 justes, la plupart des faits, et 3 actions sur 5. Ouvrez `resultats/rapport - 3 - API et playbooks.md`.

❓ **Question pour la classe :** quelles actions manquent ? Pourquoi un playbook **déterministe** ne peut-il pas les faire ?

*Réponse : la clé SSH et la crontab de l'attaquant ne sont pas retirées. Le playbook les **lit** (elles sont dans le rapport), mais il ne sait pas **laquelle** des clés est malveillante : il y a aussi la clé légitime `deploiement@ci-uqac`. Décider demande du **jugement** : ce sera le rôle de l'étape IA (exercice 6).*

## 5. Le vrai serveur

Dans un **terminal** de VS Code, à la racine du dossier `SOAR` :
```
SOAR_SECRET_WEBHOOK=cle-du-siem-8inf857 uvicorn minisoar.api:app --port 8000
```
(sous Windows PowerShell : `$env:SOAR_SECRET_WEBHOOK="cle-du-siem-8inf857"; uvicorn minisoar.api:app --port 8000`)

Ouvrez **http://localhost:8000/docs** : FastAPI génère la documentation de l'API, et vous pouvez envoyer une alerte depuis le navigateur. Ou, depuis un autre terminal :
```
curl -X POST localhost:8000/webhooks/siem -H "x-soar-cle: cle-du-siem-8inf857" -H "Content-Type: application/json" -d @environnement/une-alerte.json
```

In [8]:
import json
_ = open("environnement/une-alerte.json", "w").write(json.dumps(alertes[5], ensure_ascii=False))

## 6. À vous : une troisième source

L'entreprise installe un **EDR** (antivirus nouvelle génération). Ses alertes ressemblent à ceci :
```json
{"alert_id": "edr-5521", "created": "2016-12-10T09:20:00Z",
 "detection": {"name": "Cryptominer", "severity": "high"},
 "device": {"hostname": "LabSZ"}, "process": {"user": "webadmin", "cmdline": "/tmp/.x/kworkerds -o ..."}}
```
Dans `normaliser` (fichier `api.py`), ajoutez la source `"edr"` : la détection `Cryptominer` devient la règle `R5-cpu` ; la sévérité `low`/`medium`/`high`/`critical` devient le niveau 3/6/10/14 ; la description reprend `detection.name` et `process.cmdline`. Redémarrez le noyau et lancez le test.

In [ ]:
from minisoar import api
edr = {"alert_id": "edr-5521", "created": "2016-12-10T09:20:00Z",
       "detection": {"name": "Cryptominer", "severity": "high"},
       "device": {"hostname": "LabSZ"}, "process": {"user": "webadmin", "cmdline": "/tmp/.x/kworkerds -o 198.51.100.9:3333"}}
a = api.normaliser("edr", edr)
assert (a.regle, a.niveau, a.compte, a.hote) == ("R5-cpu", 10, "webadmin", "LabSZ"), a
assert "kworkerds" in a.description
print("✅", a)

*Réponse :*
```python
    if source == "edr":
        regles = {"Cryptominer": "R5-cpu"}
        niveaux = {"low": 3, "medium": 6, "high": 10, "critical": 14}
        d = brut["detection"]
        return Alerte(id=brut["alert_id"], horodatage=brut["created"][:19],
                      regle=regles.get(d["name"], "R0-edr-inconnue"), niveau=niveaux[d["severity"]],
                      hote=brut["device"]["hostname"], compte=brut["process"]["user"],
                      description=f"{d['name']} : {brut['process']['cmdline']}")
```

## Ce qu'il faut retenir
- Un SOAR reçoit ses alertes par des **webhooks** authentifiés (clé d'API).
- Chaque outil a son format : on **normalise une fois**, à l'entrée, vers un format commun **validé**.
- On **dédoublonne** par empreinte, sans jamais effacer : un doublon ignoré reste compté.
- Premier rejeu complet : les playbooks déterministes font déjà l'essentiel. Ce qui manque demande du **jugement**.
- Prochain exercice : que se passe-t-il si le SOAR **plante** au milieu d'un playbook ?